# Storm growth: cloud-top cooling rate from a GOES loop

In [ ]:
%load_ext mcidasv_jupyter
%mcv_connect /path/to/runMcV
%mcv_replay off

## 1. Extract a temperature loop

In [ ]:
import mcidasv_jupyter as mcv
import numpy as np
session = mcv.get_session()
N = 5

fields = session.extract_fields('''
adde = dict(server='adde.ucar.edu', dataset='EAST', descriptor='CONUSC13',
            size='ALL', unit='TEMP', mag=(-8, -8))
frames = [loadADDEImage(position=p, **adde) for p in range(-(N-1), 1)]
panel = buildWindow(height=400, width=500)
layer = panel[0].createLayer('Image Sequence Display', frames)
''', values={'N': N}, times=range(N))

cube = np.stack([f.masked() for f in fields])
nav = fields[0]
print(cube.shape, nav.unit)

## 2. Cooling rate (K per step) — negative = growing convection

In [ ]:
d = np.diff(cube, axis=0)
cooling = np.nanmin(d, axis=0)
print('strongest cooling: %.1f K/step' % np.nanmin(cooling))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
im = ax[0].imshow(cube[-1], cmap='inferno_r'); ax[0].set_title('Tb (K), last frame')
ax[0].axis('off'); fig.colorbar(im, ax=ax[0], fraction=0.046)
im2 = ax[1].imshow(cooling, cmap='coolwarm_r', vmin=-20, vmax=20)
ax[1].set_title('max cooling per step (K)'); ax[1].axis('off')
fig.colorbar(im2, ax=ax[1], fraction=0.046)
plt.tight_layout()

## 3. Where is it growing fastest?

In [ ]:
g = np.where(np.isfinite(cooling), cooling, np.inf)
r, c = np.unravel_index(np.argmin(g), g.shape)
print('fastest growth %.1f K/step at %.2f N, %.2f E' % (cooling[r, c], nav.lats[r, c], nav.lons[r, c]))

## 4. Animate the temperature loop in McIDAS-V

In [ ]:
from scipy.interpolate import griddata

def regrid(field, values, nlat=180, nlon=300, fill=np.nan):
    m = field.valid & np.isfinite(values)
    pts = np.column_stack([field.lats[m], field.lons[m]])
    glats = np.linspace(np.nanmax(field.lats[m]), np.nanmin(field.lats[m]), nlat)
    glons = np.linspace(np.nanmin(field.lons[m]), np.nanmax(field.lons[m]), nlon)
    GLA, GLO = np.meshgrid(glats, glons, indexing='ij')
    g = griddata(pts, values[m], (GLA, GLO), method='linear')
    return np.where(np.isfinite(g), g, fill).astype('f4'), glats, glons

frames_ll = []
for f in fields:
    g, glats, glons = regrid(f, f.masked(), nlat=140, nlon=220, fill=300.0)
    frames_ll.append(g)
cube_ll = np.stack(frames_ll)

import os
OUTDIR = 'output'
os.makedirs(OUTDIR, exist_ok=True)
session.animate_grid(cube_ll, glats, glons, name='Tb',
                     out=os.path.join(OUTDIR, 'storm_growth.gif'), fps=3,
                     setup="layer.setEnhancement('ABI IR Temperature', range=(200, 300))")